# Translation Migration Audit Workflow

This notebook demonstrates how to inspect SDK source before translating it. It uses the committed translation example corpus to compare accepted snippets, rejected patterns, portable rewrites, semantic contracts, migration-audit reports, round-trip verification, result normalization, and purpose-level workflows.


## Problem

A user may have a circuit or workflow written in one SDK and need to know whether it can move to another free local SDK without changing purpose. The safest workflow is to audit the source first, then translate only if the source fits the declared neutral semantic subset.


## Translation Scope

The package is intentionally conservative. `translate-check` reports what is preserved, what is rewritten as SDK syntax, what would be rejected, and which behavior is not modeled. Round-trip verification then checks translated source by reimporting it through the neutral model.

## Variables and Parameters

- `AUDIT_SOURCE`: Qiskit source file used for the migration-audit walkthrough.
- `TARGET_FORMAT`: target local SDK format for the preflight audit, here `cirq`.
- `EXAMPLE_DIR`: checked-in translation fixture directory.
- `ARTIFACT_DIR`: notebook artifact directory for generated reports and translated source.
- `result_cases`: SDK-shaped result JSON fixtures normalized into neutral result JSON.


## Setup

Use only public package APIs and committed example fixtures. No cloud credentials, paid providers, or optional SDK runtimes are required for these checks.


In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from quantum_backend_bench.core.circuit_translate import (
    TranslationError,
    import_circuit_source,
    translate_circuit_source,
    translation_check_report,
    translation_error_report,
    translation_result_report,
)
from quantum_backend_bench.core.workflow_translate import (
    normalize_result_source,
    translate_workflow_source,
)
from quantum_backend_bench.utils.notebook import notebook_artifact_dir, verification_frame

ARTIFACT_DIR = notebook_artifact_dir()
REPO_ROOT = Path.cwd().resolve()
for candidate in (REPO_ROOT, *REPO_ROOT.parents):
    if (candidate / "examples" / "translation").exists():
        REPO_ROOT = candidate
        break

EXAMPLE_DIR = REPO_ROOT / "examples" / "translation"
AUDIT_SOURCE = EXAMPLE_DIR / "migration_audit" / "qiskit_static_bell.py"
TARGET_FORMAT = "cirq"


def source_preview(title, source, max_lines=30):
    lines = source.strip().splitlines()
    print(title)
    print("-" * len(title))
    for number, line in enumerate(lines[:max_lines], start=1):
        print(f"{number:>2}: {line}")
    if len(lines) > max_lines:
        print(f"... {len(lines) - max_lines} more lines")


def key_value_frame(mapping):
    return pd.DataFrame([{"field": key, "value": value} for key, value in mapping.items()])


def migration_audit_frame(report):
    audit = report["migration_audit"]
    rows = []
    for field in ("preserved", "rewritten", "rejected_if_present", "not_modeled"):
        for item in audit[field]:
            rows.append({"category": field, "detail": item})
    rows.append({"category": "verification", "detail": audit["verification_recommendation"]})
    return pd.DataFrame(rows)

## Source Under Review

Start with a small static Qiskit circuit. It is intentionally simple so the audit output is easy to inspect.


In [ ]:
audit_source = AUDIT_SOURCE.read_text(encoding="utf-8")
source_preview("Qiskit source under review", audit_source)

## Semantic Contract and Migration Audit

The report is target-aware because `to_format` is provided. The semantic contract is the high-level promise; the migration audit is the practical checklist for this source/target pair.


In [ ]:
benchmark, detected_format = import_circuit_source(audit_source, from_format="qiskit")
audit_report = translation_check_report(
    benchmark,
    detected_format,
    source_path=str(AUDIT_SOURCE),
    to_format=TARGET_FORMAT,
)

contract = audit_report["semantic_contract"]
audit = audit_report["migration_audit"]

display(
    key_value_frame(
        {
            "input format": audit_report["input_format"],
            "target": audit["target"],
            "guarantee": contract["guarantee"],
            "operations": audit["operation_count"],
            "gate counts": json.dumps(audit["gate_counts"], sort_keys=True),
        }
    )
)
display(migration_audit_frame(audit_report))

expected_report_path = (
    EXAMPLE_DIR / "migration_audit" / "expected" / "qiskit_static_bell_to_cirq_check.json"
)
expected_report = json.loads(expected_report_path.read_text(encoding="utf-8"))
assert expected_report["migration_audit"]["gate_counts"] == audit["gate_counts"]

## Accepted, Rejected, and Portable Examples

A good migration review needs both positive and negative examples. Accepted fixtures demonstrate current support. Rejected fixtures pin diagnostics. Portable rewrites show how to move unsupported patterns into the neutral subset.


In [ ]:
example_cases = [
    ("accepted", EXAMPLE_DIR / "accepted" / "qiskit_static_rotations.py", "qiskit"),
    ("rejected", EXAMPLE_DIR / "rejected" / "custom_gate_qiskit.py", "qiskit"),
    ("portable rewrite", EXAMPLE_DIR / "portable" / "custom_gate_decomposed_qiskit.py", "qiskit"),
]

example_rows = []
for label, path, from_format in example_cases:
    try:
        benchmark, detected = import_circuit_source(
            path.read_text(encoding="utf-8"), from_format=from_format
        )
        report = translation_check_report(
            benchmark, detected, source_path=str(path), to_format="cirq"
        )
        example_rows.append(
            {
                "case": label,
                "file": path.name,
                "status": report["migration_audit"]["status"],
                "diagnostics": "-",
            }
        )
    except TranslationError as exc:
        report = translation_error_report(exc, source_path=str(path), from_format=from_format)
        example_rows.append(
            {
                "case": label,
                "file": path.name,
                "status": report["status"],
                "diagnostics": ", ".join(item["code"] for item in report["diagnostics"]),
            }
        )

display(pd.DataFrame(example_rows))

## Verified Round Trip

After preflight passes, translate the source and verify exact neutral probabilities. The saved report is comparable to the committed `roundtrip_audit/expected/` artifacts.


In [ ]:
translation_result = translate_circuit_source(
    audit_source,
    from_format="qiskit",
    to_format=TARGET_FORMAT,
    verify="exact",
)
roundtrip_report = translation_result_report(
    translation_result,
    source_path=str(AUDIT_SOURCE),
    from_format="qiskit",
    to_format=TARGET_FORMAT,
)

source_path = ARTIFACT_DIR / "qiskit_static_bell_to_cirq.py"
report_path = ARTIFACT_DIR / "qiskit_static_bell_to_cirq_roundtrip.json"
source_path.write_text(translation_result.source, encoding="utf-8")
report_path.write_text(
    json.dumps(roundtrip_report, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)

verification = roundtrip_report["verification"]
display(
    key_value_frame(
        {
            "translated source": str(source_path),
            "report": str(report_path),
            "verification mode": verification["mode"],
            "passed": verification["passed"],
            "TVD": verification["total_variation_distance"],
        }
    )
)

## Result Normalization Edge Cases

Migration is not only source generation. SDK-shaped result payloads also need normalization so downstream analysis can compare counts and probabilities consistently.


In [ ]:
result_cases = [
    ("qiskit-counts-json", EXAMPLE_DIR / "results" / "qiskit_spaced_counts_no_shots.json"),
    ("cirq-counts-json", EXAMPLE_DIR / "results" / "cirq_multi_key_counts.json"),
    ("pennylane-samples-json", EXAMPLE_DIR / "results" / "pennylane_nested_samples.json"),
    ("braket-counts-json", EXAMPLE_DIR / "results" / "braket_counts_fallback.json"),
]

normalized_rows = []
for from_format, path in result_cases:
    result = normalize_result_source(path.read_text(encoding="utf-8"), from_format=from_format)
    payload = json.loads(result.source)
    normalized_rows.append(
        {
            "from_format": from_format,
            "file": path.name,
            "shots": payload["shots"],
            "states": ", ".join(sorted(payload["counts"])),
            "probability_sum": round(sum(payload["probabilities"].values()), 12),
        }
    )

display(pd.DataFrame(normalized_rows))

## Purpose Workflow Snapshot

Purpose-level `workflow-json` examples encode sampler, estimator, parameter-sweep-point, and QAOA-style intent. They are a better migration contract than arbitrary Python when a workflow has execution settings and measurement requests.


In [ ]:
purpose_cases = [
    ("sampler", EXAMPLE_DIR / "purpose_workflows" / "sampler_workflow.json", "cirq"),
    ("estimator", EXAMPLE_DIR / "purpose_workflows" / "estimator_workflow.json", "qiskit_aer"),
    (
        "parameter sweep point",
        EXAMPLE_DIR / "purpose_workflows" / "parameter_sweep_workflow.json",
        "pennylane",
    ),
    ("qaoa", EXAMPLE_DIR / "purpose_workflows" / "qaoa_workflow.json", "braket_local"),
]

purpose_rows = []
for name, path, target in purpose_cases:
    result = translate_workflow_source(
        path.read_text(encoding="utf-8"),
        from_format="workflow-json",
        to_format=target,
        verify="canonical",
    )
    purpose_rows.append(
        {
            "purpose": name,
            "target": target,
            "verified": result.verification.passed if result.verification else None,
            "source_lines": len(result.source.splitlines()),
        }
    )

display(pd.DataFrame(purpose_rows))

## Verification Summary

The checks below keep the notebook focused on actionable migration review: preflight support, exact round-trip verification, normalized probability sums, and purpose-workflow verification.


In [ ]:
checks = [
    {
        "check": "migration audit source supported",
        "value": audit["status"],
        "expected": "target_supported",
        "passed": audit["status"] == "target_supported",
    },
    {
        "check": "round-trip exact verification",
        "value": verification["total_variation_distance"],
        "expected": "<= 1e-09",
        "passed": verification["passed"],
    },
]
checks.extend(
    {
        "check": f"{row['from_format']} probability sum",
        "value": row["probability_sum"],
        "expected": 1.0,
        "passed": row["probability_sum"] == 1.0,
    }
    for row in normalized_rows
)
checks.extend(
    {
        "check": f"{row['purpose']} purpose workflow",
        "value": row["verified"],
        "expected": True,
        "passed": row["verified"] is True,
    }
    for row in purpose_rows
)
display(verification_frame(checks))